In [3]:
pip install pyspark


Note: you may need to restart the kernel to use updated packages.


## Crear la SparkSession y cargar el CSV ##

In [4]:
# Arrancamos Spark en local (sin clúster externo)
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HotelReviews") \
    .master("local[*]") \
    .getOrCreate()

# Cargamos el CSV indicando cabecera y que deduzca los tipos
df_spark_sql = spark.read.csv(
    "Hotel_Reviews.csv",
    header=True,
    inferSchema=True
)

# Vista rápida de los primeros registros y del esquema
df_spark_sql.show(5)
df_spark_sql.printSchema()

+--------------------+----------------------------+-----------+-------------+-----------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+------------------------------------------+--------------+--------------------+-----------------+----------+---------+
|       Hotel_Address|Additional_Number_of_Scoring|Review_Date|Average_Score| Hotel_Name|Reviewer_Nationality|     Negative_Review|Review_Total_Negative_Word_Counts|Total_Number_of_Reviews|     Positive_Review|Review_Total_Positive_Word_Counts|Total_Number_of_Reviews_Reviewer_Has_Given|Reviewer_Score|                Tags|days_since_review|       lat|      lng|
+--------------------+----------------------------+-----------+-------------+-----------+--------------------+--------------------+---------------------------------+-----------------------+--------------------+---------------------------------+------------------------------

## DataFrames en Spark: SparkSQL. ##

In [9]:
# Usamos funciones nativas de Spark para evitar UDFs
from pyspark.sql.functions import col, when, regexp_replace

# Creamos una categoría textual a partir de la puntuación media
df_spark_sql = df_spark_sql.withColumn(
    "score_string",
    when(col("Average_Score") < 5, "Bad")
    .when(col("Average_Score") < 7, "Normal")
    .when(col("Average_Score") < 9, "Good")
    .when(col("Average_Score") < 10, "Excellent")
    .otherwise("Perfect")
)

# Asignamos un valor entero a cada categoría, que será la etiqueta del modelo
df_spark_sql = df_spark_sql.withColumn(
    "score_evaluation",
    when(col("score_string") == "Bad", 0)
    .when(col("score_string") == "Normal", 1)
    .when(col("score_string") == "Good", 2)
    .when(col("score_string") == "Excellent", 3)
    .otherwise(4)   # Perfect
)

# Eliminamos el sufijo de texto en "days_since_review" y lo dejamos como entero
df_spark_sql = df_spark_sql.withColumn(
    "days_since_review",
    regexp_replace(col("days_since_review"), " days?", "")  # quita ' day' o ' days'
).withColumn(
    "days_since_review",
    col("days_since_review").cast("int")
)


### Ejercicio 1: Crear un bucle que muestre todas las columnas del DataFrame, junto con sus tipos. También puedes pintar el esquema del Dataframe. ###

In [10]:
for name, dtype in df_spark_sql.dtypes:
    print(f"{name}: {dtype}")

df_spark_sql.printSchema()


Hotel_Address: string
Additional_Number_of_Scoring: int
Review_Date: string
Average_Score: double
Hotel_Name: string
Reviewer_Nationality: string
Negative_Review: string
Review_Total_Negative_Word_Counts: int
Total_Number_of_Reviews: int
Positive_Review: string
Review_Total_Positive_Word_Counts: int
Total_Number_of_Reviews_Reviewer_Has_Given: int
Reviewer_Score: double
Tags: string
days_since_review: int
lat: string
lng: string
score_string: string
score_evaluation: int
root
 |-- Hotel_Address: string (nullable = true)
 |-- Additional_Number_of_Scoring: integer (nullable = true)
 |-- Review_Date: string (nullable = true)
 |-- Average_Score: double (nullable = true)
 |-- Hotel_Name: string (nullable = true)
 |-- Reviewer_Nationality: string (nullable = true)
 |-- Negative_Review: string (nullable = true)
 |-- Review_Total_Negative_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews: integer (nullable = true)
 |-- Positive_Review: string (nullable = true)
 |-- Review_Tota

### Ejercicio 2: Realizar un muestreo de 10 valores únicos de nombres de hoteles. Ordénalos alfanuméricamente de forma ascendente (primero los números 0-9, después A-Z). ###

In [11]:
from pyspark.sql.functions import col

(df_spark_sql
    .select("Hotel_Name")
    .distinct()
    .orderBy(col("Hotel_Name").asc())
    .limit(10)
    .show(truncate=False))


+---------------------------------------------------+
|Hotel_Name                                         |
+---------------------------------------------------+
|11 Cadogan Gardens                                 |
|1K Hotel                                           |
|25hours Hotel beim MuseumsQuartier                 |
|41                                                 |
|45 Park Lane Dorchester Collection                 |
|88 Studios                                         |
|9Hotel Republique                                  |
|A La Villa Madame                                  |
|ABaC Restaurant Hotel Barcelona GL Monumento       |
|AC Hotel Barcelona Forum a Marriott Lifestyle Hotel|
+---------------------------------------------------+



### Ejercicio 3: Transforma las columnas *lat* y *lng* al tipo Float.

In [12]:
from pyspark.sql.functions import expr

# Convertimos lat/lng a float, cambiando el literal 'NA' por null
df_spark_sql = (
    df_spark_sql
    .withColumn("lat", when(col("lat") == "NA", None).otherwise(col("lat")).cast("float"))
    .withColumn("lng", when(col("lng") == "NA", None).otherwise(col("lng")).cast("float"))
)

df_spark_sql.select("lat", "lng").printSchema()


root
 |-- lat: float (nullable = true)
 |-- lng: float (nullable = true)



In [13]:
# Dividimos el dataset en entrenamiento y test (67% / 33%) con semilla fija
splits = df_spark_sql.randomSplit([0.67, 0.33], seed=42)

# Eliminamos filas con nulos para evitar problemas en ML
df_spark_sql_train = splits[0].dropna()
df_spark_sql_test  = splits[1].dropna()

print("Registros train:", df_spark_sql_train.count())
print("Registros test :", df_spark_sql_test.count())

Registros train: 344158
Registros test : 168312


### Ejercicio 4: ¿Cuántos hoteles tienen una puntuación de 'Perfect'? ¿Y 'Good'? ¿Y 'Normal' junto a 'Good'? (Utilizar el dataset de Train)

In [14]:
from pyspark.sql.functions import col

# Contamos hoteles distintos con puntuación 'Perfect'
hoteles_perfect = (
    df_spark_sql_train
    .filter(col("score_string") == "Perfect")
    .select("Hotel_Name").distinct().count()
)

# Contamos hoteles distintos con puntuación 'Good'
hoteles_good = (
    df_spark_sql_train
    .filter(col("score_string") == "Good")
    .select("Hotel_Name").distinct().count()
)

# Contamos hoteles distintos con puntuación 'Normal' o 'Good'
hoteles_normal_good = (
    df_spark_sql_train
    .filter(col("score_string").isin("Normal", "Good"))
    .select("Hotel_Name").distinct().count()
)

print("Hoteles Perfect      :", hoteles_perfect)
print("Hoteles Good         :", hoteles_good)
print("Hoteles Normal/Good  :", hoteles_normal_good)


Hoteles Perfect      : 0
Hoteles Good         : 1164
Hoteles Normal/Good  : 1176


### Ejercicio 5: Obtener los hoteles con mayor puntuación media, descartando todos los que tengan una puntuación por encima de Good. (Utilizar el dataset de Train) ###

In [15]:
from pyspark.sql.functions import avg

# Calculamos la puntuación media por hotel usando la variable entera
puntuacion_media = (
    df_spark_sql_train
    .groupBy("Hotel_Name")
    .agg(avg("score_evaluation").alias("avg_score_eval"))
)

# Nos quedamos con los hoteles cuya media no supera 'Good' (2)
hoteles_filtrados = (
    puntuacion_media
    .filter(col("avg_score_eval") <= 2)
    .orderBy(col("avg_score_eval").desc())
)

hoteles_filtrados.show(20, truncate=False)


+------------------------------------------------------+--------------+
|Hotel_Name                                            |avg_score_eval|
+------------------------------------------------------+--------------+
|Select Hotel                                          |2.0           |
|H tel Chaplain Paris Rive Gauche                      |2.0           |
|Hotel Louvre Montana                                  |2.0           |
|Le Senat                                              |2.0           |
|Pullman London St Pancras                             |2.0           |
|H tel De Vend me                                      |2.0           |
|Hotel Panache                                         |2.0           |
|H tel Duo                                             |2.0           |
|Ace Hotel London Shoreditch                           |2.0           |
|The Park Tower Knightsbridge a Luxury Collection Hotel|2.0           |
|Park Plaza Sherlock Holmes London                     |2.0     


# Machine Learning en Apache Spark: Spark MLLib y Spark ML #

## Clasificación Supervisada: Árboles de decisión ##

### Ejercicio 6.1: Volver a observar todas las columnas del dataframe, para identificar las que sean categóricas. ###

In [16]:

df_spark_sql_train.printSchema()

# Listamos las columnas de tipo string para identificar las categóricas
[c for c, t in df_spark_sql_train.dtypes if t == "string"]


root
 |-- Hotel_Address: string (nullable = true)
 |-- Additional_Number_of_Scoring: integer (nullable = true)
 |-- Review_Date: string (nullable = true)
 |-- Average_Score: double (nullable = true)
 |-- Hotel_Name: string (nullable = true)
 |-- Reviewer_Nationality: string (nullable = true)
 |-- Negative_Review: string (nullable = true)
 |-- Review_Total_Negative_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews: integer (nullable = true)
 |-- Positive_Review: string (nullable = true)
 |-- Review_Total_Positive_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews_Reviewer_Has_Given: integer (nullable = true)
 |-- Reviewer_Score: double (nullable = true)
 |-- Tags: string (nullable = true)
 |-- days_since_review: integer (nullable = true)
 |-- lat: float (nullable = true)
 |-- lng: float (nullable = true)
 |-- score_string: string (nullable = false)
 |-- score_evaluation: integer (nullable = false)



['Hotel_Address',
 'Review_Date',
 'Hotel_Name',
 'Reviewer_Nationality',
 'Negative_Review',
 'Positive_Review',
 'Tags',
 'score_string']

### Ejercicio 6.2: Eliminar, de los dataframes df_spark_sql_train y df_spark_sql test, las variables 'Hotel_Address', 'Hotel_Name', 'Tags', 'Positive Review', 'Negative_Review' y 'score_string'. Llamarlos: df_DT_train y df_DT_test. ### 

In [17]:
# Quitamos textos largos y campos que no se van a usar como predictores
cols_a_eliminar = [
    c for c in [
        "Hotel_Address", "Hotel_Name", "Tags",
        "Positive Review", "Positive_Review",
        "Negative Review", "Negative_Review",
        "score_string"
    ]
    if c in df_spark_sql_train.columns
]

df_DT_train = df_spark_sql_train.drop(*cols_a_eliminar)
df_DT_test  = df_spark_sql_test.drop(*cols_a_eliminar)

df_DT_train.printSchema()


root
 |-- Additional_Number_of_Scoring: integer (nullable = true)
 |-- Review_Date: string (nullable = true)
 |-- Average_Score: double (nullable = true)
 |-- Reviewer_Nationality: string (nullable = true)
 |-- Review_Total_Negative_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews: integer (nullable = true)
 |-- Review_Total_Positive_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews_Reviewer_Has_Given: integer (nullable = true)
 |-- Reviewer_Score: double (nullable = true)
 |-- days_since_review: integer (nullable = true)
 |-- lat: float (nullable = true)
 |-- lng: float (nullable = true)
 |-- score_evaluation: integer (nullable = false)



### Ejercicio 7: Para cada columa restante que sea String ('Review_Date' y 'Review_Nationality'), aplicar un StringIndexer(), devolviendo como resultado la misma columna, pero con su nombre acabando en _index. Sobreescribir ambos dataframes.  ###

In [18]:
from pyspark.ml.feature import StringIndexer

# Detectamos qué columnas siguen siendo string (por ejemplo fecha y nacionalidad)
columnas_string = [c for c, t in df_DT_train.dtypes if t == "string"]

# Indexamos cada columna categórica, creando una columna *_index
for colname in columnas_string:
    indexer = StringIndexer(
        inputCol=colname,
        outputCol=f"{colname}_index",
        handleInvalid="keep"  # así no falla si en test aparece una categoría nueva
    ).fit(df_DT_train)

    df_DT_train = indexer.transform(df_DT_train)
    df_DT_test  = indexer.transform(df_DT_test)

df_DT_train.select([c for c in df_DT_train.columns if c.endswith("_index")]).show(5)


+-----------------+--------------------------+
|Review_Date_index|Reviewer_Nationality_index|
+-----------------+--------------------------+
|            582.0|                      37.0|
|            116.0|                       0.0|
|            116.0|                       0.0|
|            707.0|                       0.0|
|             40.0|                       3.0|
+-----------------+--------------------------+
only showing top 5 rows


### Ejercicio 8: Aplicar VectorAssembler() sobre las columnas que no son ni las dos anteriores, ni la columna 'score_evaluation', devolviendo una columna llamada 'features'. Llamar al resultado DT_vector_assembler. ###

In [19]:
from pyspark.ml.feature import VectorAssembler

# Seleccionamos columnas numéricas que servirán como predictores
numeric_types = ("int", "double", "float", "bigint", "smallint", "tinyint", "decimal")
numeric_cols = [
    c for c, t in df_DT_train.dtypes
    if t in numeric_types and c != "score_evaluation"  # la etiqueta se excluye
]

# Añadimos las columnas indexadas creadas anteriormente
index_cols = [f"{c}_index" for c in columnas_string]

# Esta lista es la entrada final al modelo
feature_cols = numeric_cols + index_cols

DT_vector_assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)


### Ejercicio 9: Aplicar el transformador sobre ambos dataframes. ###

In [20]:
# Añadimos la columna 'features' a los conjuntos de train y test
df_DT_train = DT_vector_assembler.transform(df_DT_train)
df_DT_test  = DT_vector_assembler.transform(df_DT_test)

df_DT_train.select("features", "score_evaluation").show(5, truncate=False)


+----------------------------------------------------------------------------------------------------+----------------+
|features                                                                                            |score_evaluation|
+----------------------------------------------------------------------------------------------------+----------------+
|[194.0,7.7,0.0,1403.0,23.0,15.0,6.3,570.0,52.36057662963867,4.915968418121338,582.0,37.0,582.0,37.0]|2               |
|[194.0,7.7,103.0,1403.0,42.0,1.0,7.1,569.0,52.36057662963867,4.915968418121338,116.0,0.0,116.0,0.0] |2               |
|[194.0,7.7,0.0,1403.0,13.0,1.0,9.6,569.0,52.36057662963867,4.915968418121338,116.0,0.0,116.0,0.0]   |2               |
|[194.0,7.7,60.0,1403.0,15.0,7.0,7.9,566.0,52.36057662963867,4.915968418121338,707.0,0.0,707.0,0.0]  |2               |
|[194.0,7.7,9.0,1403.0,0.0,1.0,5.4,562.0,52.36057662963867,4.915968418121338,40.0,3.0,40.0,3.0]      |2               |
+---------------------------------------

### Ejercicio 10: Inicializar el modelo de árbol de decisión, entrenarlo y aplicarlo sobre los datos de test. ###
* Modelo: DecisionTreeClassifier:
  * Label: score_evaluation.
  * Features: features.
  * maxBins: 1000
  * maxDepth: 1

In [21]:
from pyspark.ml.classification import DecisionTreeClassifier

# Definimos un árbol poco profundo para ver el comportamiento básico del modelo
DT_classifier = DecisionTreeClassifier(
    labelCol="score_evaluation",
    featuresCol="features",
    maxBins=1000,   # permite discretizar muchas categorías
    maxDepth=1      # solo una división: modelo muy simple
)

# Ajustamos el modelo sobre el conjunto de entrenamiento
DT_model = DT_classifier.fit(df_DT_train)

# Generamos predicciones sobre el conjunto de test
DT_predictions = DT_model.transform(df_DT_test)

DT_predictions.select("score_evaluation", "prediction", "probability").show(10, truncate=False)


+----------------+----------+-------------------------------------------------+
|score_evaluation|prediction|probability                                      |
+----------------+----------+-------------------------------------------------+
|2               |2.0       |[0.0,0.013095074298293891,0.9869049257017061,0.0]|
|2               |2.0       |[0.0,0.013095074298293891,0.9869049257017061,0.0]|
|2               |2.0       |[0.0,0.013095074298293891,0.9869049257017061,0.0]|
|2               |2.0       |[0.0,0.013095074298293891,0.9869049257017061,0.0]|
|2               |2.0       |[0.0,0.013095074298293891,0.9869049257017061,0.0]|
|2               |2.0       |[0.0,0.013095074298293891,0.9869049257017061,0.0]|
|2               |2.0       |[0.0,0.013095074298293891,0.9869049257017061,0.0]|
|2               |2.0       |[0.0,0.013095074298293891,0.9869049257017061,0.0]|
|2               |2.0       |[0.0,0.013095074298293891,0.9869049257017061,0.0]|
|2               |2.0       |[0.0,0.0130

### Ejercicio 11: Evaluar el modelo aplicándole un clasificador multiclase. Calcular la métrica 'accuracy', y conseguir el complementario para calcular el error. ###
* Evaluador: MulticlassClassificationEvaluator
  * Label: score_evaluation.
  * Prediction: prediction.
  * MetricName: accuracy.

In [22]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Usamos accuracy como métrica de evaluación
evaluator = MulticlassClassificationEvaluator(
    labelCol="score_evaluation",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(DT_predictions)
error = 1.0 - accuracy

print("Accuracy:", accuracy)
print("Error   :", error)


Accuracy: 0.9889609772327582
Error   : 0.01103902276724178


## Spark ML: Pipelines ##

### Pipelines: Árboles de Decisión ###
Con el mismo concepto que con el KMeans, se va a diseñar el flujo para los árboles de decisión. Primero hay que aplicar los cambios de preprocesamiento vistos anteriormente al DataFrame inicial para preparalo.

### Ejercicio 12: Eliminar, de los dataframes df_spark_sql_train y df_spark_sql test, las variables 'Hotel_Address', 'Hotel_Name', 'Tags', 'Positive Review', 'Negative_Review' y 'score_string'. Llamarlos: df_DT_train y df_DT_test. ### 

In [23]:
# Volvemos a preparar un dataset limpio a partir de train/test originales
cols_a_eliminar_pipeline = [
    c for c in [
        "Hotel_Address", "Hotel_Name", "Tags",
        "Positive Review", "Positive_Review",
        "Negative Review", "Negative_Review",
        "score_string"
    ]
    if c in df_spark_sql_train.columns
]

df_DT_train = df_spark_sql_train.drop(*cols_a_eliminar_pipeline)
df_DT_test  = df_spark_sql_test.drop(*cols_a_eliminar_pipeline)

df_DT_train.printSchema()



root
 |-- Additional_Number_of_Scoring: integer (nullable = true)
 |-- Review_Date: string (nullable = true)
 |-- Average_Score: double (nullable = true)
 |-- Reviewer_Nationality: string (nullable = true)
 |-- Review_Total_Negative_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews: integer (nullable = true)
 |-- Review_Total_Positive_Word_Counts: integer (nullable = true)
 |-- Total_Number_of_Reviews_Reviewer_Has_Given: integer (nullable = true)
 |-- Reviewer_Score: double (nullable = true)
 |-- days_since_review: integer (nullable = true)
 |-- lat: float (nullable = true)
 |-- lng: float (nullable = true)
 |-- score_evaluation: integer (nullable = false)



Después se diseña el flujo para este modelo, el cual será:

** StringIndexer --> VectorAssembler --> Decission Tree (Inicialización) --> Decission Tree (Entrenamiento) --> Modelo Decission Tree entrenado ** 

 ### Ejercicio 13: Recoger una lista con todos los StringIndexer a aplicar, y llamarla DT_string_indexers ###
 En lugar de sobreescribir cada vez el dataframe, crear una lista, y con el método 'append', se irán añadiendo todos los StringIndexers().

In [24]:
from pyspark.ml.feature import StringIndexer

# Guardamos los indexadores como etapas del pipeline en lugar de aplicarlos ya
DT_string_indexers = []

# Identificamos de nuevo las columnas categóricas en la versión limpia
columnas_string = [c for c, t in df_DT_train.dtypes if t == "string"]

for colname in columnas_string:
    DT_string_indexers.append(
        StringIndexer(
            inputCol=colname,
            outputCol=f"{colname}_index",
            handleInvalid="keep"  # mantenemos etiquetas nuevas en una categoría extra
        )
    )

DT_string_indexers


[StringIndexer_b00b3b6c843c, StringIndexer_93f69382aedc]

### Ejercicio 14: Guardar en la variable 'DT_vector_assembler' la aplicación del mismo VectorAssembler() del ejercicio 8. ###

In [25]:
from pyspark.ml.feature import VectorAssembler

# Recalculamos las columnas numéricas de entrada en la versión limpia
numeric_types = ("int", "double", "float", "bigint", "smallint", "tinyint", "decimal")
numeric_cols = [
    c for c, t in df_DT_train.dtypes
    if t in numeric_types and c != "score_evaluation"
]

# Añadimos las columnas indexadas que generarán los StringIndexers
index_cols = [f"{c}_index" for c in columnas_string]

feature_cols = numeric_cols + index_cols

DT_vector_assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

feature_cols

['Additional_Number_of_Scoring',
 'Average_Score',
 'Review_Total_Negative_Word_Counts',
 'Total_Number_of_Reviews',
 'Review_Total_Positive_Word_Counts',
 'Total_Number_of_Reviews_Reviewer_Has_Given',
 'Reviewer_Score',
 'days_since_review',
 'lat',
 'lng',
 'Review_Date_index',
 'Reviewer_Nationality_index']

### Ejercicio 15: Crear una lista con el mombre de DT_pipeline_stages, y añadirle la lista de StringIndexers y el VectorAssembler (en este orden) ###

In [26]:
# Construimos la lista de etapas: primero indexadores, luego ensamblador
DT_pipeline_stages = []

DT_pipeline_stages.extend(DT_string_indexers)  # aplica los StringIndexer en orden
DT_pipeline_stages.append(DT_vector_assembler) # después crea la columna 'features'

DT_pipeline_stages


[StringIndexer_b00b3b6c843c,
 StringIndexer_93f69382aedc,
 VectorAssembler_fb03d961ecea]

### Ejercicio 16: Inicializar el modelo de árbol de decisión (mismas especificaciones que en el ej. 10), y añadirlo a la lista de pasos 'DT_pipeline_stages' ###

In [27]:
from pyspark.ml.classification import DecisionTreeClassifier

# Reutilizamos la misma configuración del árbol que antes
DT_classifier = DecisionTreeClassifier(
    labelCol="score_evaluation",
    featuresCol="features",
    maxBins=1000,
    maxDepth=1
)

# El árbol se convierte en la última etapa del pipeline
DT_pipeline_stages.append(DT_classifier)

DT_pipeline_stages

[StringIndexer_b00b3b6c843c,
 StringIndexer_93f69382aedc,
 VectorAssembler_fb03d961ecea,
 DecisionTreeClassifier_4d92cbc294ca]

### Ejercicio 17: Diseñar el Pipeline y aplicarlo sobre los datos de Train, llamándolo 'DT_pipeline_model' ###

In [28]:
from pyspark.ml import Pipeline

# Definimos el pipeline con todas las etapas encadenadas
DT_pipeline = Pipeline(stages=DT_pipeline_stages)

# Ajustamos el pipeline sobre los datos de entrenamiento
DT_pipeline_model = DT_pipeline.fit(df_DT_train)

### Ejercicio 18: Aplicar el modelo resultante sobre los datos de test y evaluarlo al igual que se hizo en el ej. 11 ###

In [29]:
# Obtenemos predicciones del pipeline sobre el conjunto de test
DT_pipeline_predictions = DT_pipeline_model.transform(df_DT_test)

# Reutilizamos el mismo evaluador de accuracy definido antes
accuracy = evaluator.evaluate(DT_pipeline_predictions)
error = 1.0 - accuracy

print("Pipeline Accuracy:", accuracy)
print("Pipeline Error   :", error)


Pipeline Accuracy: 0.9889609772327582
Pipeline Error   : 0.01103902276724178
